# NLP817: Assignment 3

In [25]:
import os
import glob
import random
from pathlib import Path
import torch
import sacrebleu
from datasets import Dataset, load_dataset
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR    = Path("ee_mt.v0.1")
MODEL_DIR   = Path("model") 
MODEL_PATH  = Path("models/opus-mt-en-af")
OPUS_FILE   = Path("data/train-00000-of-00001.parquet")

MODEL_DIR.mkdir(exist_ok=True)

## Load & Parse Provided Parallel Data


In [7]:
def parse_parallel_files(af_path, en_path):
    # Read paired .af.txt and .en.txt files, skip empty lines and % comments
    pairs = []
    with open(af_path, encoding="utf-8") as f_af, \
         open(en_path, encoding="utf-8") as f_en:
        for af_line, en_line in zip(f_af, f_en):
            af_line = af_line.strip()
            en_line = en_line.strip()
            if not af_line or not en_line:
                continue
            if af_line.startswith("%") or en_line.startswith("%"):
                continue
            pairs.append({"en": en_line, "af": af_line})
    return pairs


def load_split(split_dir):
    # Load all parallel pairs from a directory of .af.txt / .en.txt files.
    pairs = []
    af_files = sorted(glob.glob(str(split_dir / "*.af.txt")))
    for af_path in af_files:
        en_path = af_path.replace(".af.txt", ".en.txt")
        if os.path.exists(en_path):
            pairs.extend(parse_parallel_files(af_path, en_path))
    return pairs


train_provided = load_split(DATA_DIR / "train")
val_provided   = load_split(DATA_DIR / "val")

print(f"Provided train pairs: {len(train_provided)}")
print(f"Provided val pairs:   {len(val_provided)}")
print("\nSample train pair:")
print("English: ", train_provided[100]["en"])
print("Afrikaans: ", train_provided[100]["af"])

Provided train pairs: 454
Provided val pairs:   182

Sample train pair:
English:  Answer these questions and motivate your answers.
Afrikaans:  Beantwoord hierdie vrae en motiveer jou antwoorde.


## Fetch OPUS External Data

In [20]:
# Load OPUS external data from locally downloaded parquet file
print(f"Loading OPUS data from {OPUS_FILE}...")
opus_ds = load_dataset("parquet", data_files={"train": str(OPUS_FILE)}, split="train")

# Opus stores pairs here
opus_pairs = []
for item in opus_ds:
    en = item["translation"]["en"].strip()
    af = item["translation"]["af"].strip()
    if en and af:
        opus_pairs.append({"en": en, "af": af})

print(f"OPUS af-en pairs loaded: {len(opus_pairs)}")
print("\nSample OPUS pair:")
print("English: ", opus_pairs[0]["en"])
print("Afrikaans: ", opus_pairs[0]["af"])

Loading OPUS data from data/train-00000-of-00001.parquet...
OPUS af-en pairs loaded: 275512

Sample OPUS pair:
English:  Come on.
Afrikaans:  Kom


## Combine & Upsample Training Data


In [21]:
# Combine OPUS data with provided training data 
UPSAMPLE_FACTOR = 3
# Cap OPUS to keep training time reasonable on CPU
MAX_OPUS_PAIRS = 50000  


random.shuffle(opus_pairs)
opus_capped = opus_pairs[:MAX_OPUS_PAIRS]

train_upsampled = train_provided * UPSAMPLE_FACTOR

train_all = opus_capped + train_upsampled
random.shuffle(train_all)

print(f"OPUS pairs (capped):        {len(opus_capped)}")
print(f"Provided train (upsampled): {len(train_upsampled)}")
print(f"Total training pairs:       {len(train_all)}")
print(f"Validation pairs:           {len(val_provided)}")

OPUS pairs (capped):        50000
Provided train (upsampled): 1362
Total training pairs:       51362
Validation pairs:           182


## Load Pretrained Model & Tokenizer


In [22]:
# Load pretrained OPUS model and tokenizer from local files
print(f"Loading tokenizer and model from {MODEL_PATH}")
tokenizer = MarianTokenizer.from_pretrained(str(MODEL_PATH))
model = MarianMTModel.from_pretrained(str(MODEL_PATH))

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Tokenizer vocab size:", tokenizer.vocab_size)

sample_en = "Calculate the voltage across the resistor."
inputs = tokenizer([sample_en], return_tensors="pt", padding=True)
translated = model.generate(**inputs)
sample_translation = tokenizer.decode(translated[0], skip_special_tokens=True)
print(f"\nPretrained translation test:")
print(f" English: {sample_en}")
print(f" Afrikaans: {sample_translation}")

Loading tokenizer and model from models/opus-mt-en-af


/opt/homebrew/Caskroom/miniforge/base/envs/mlai/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 69185.50it/s]


Model parameters: 74,074,624
Tokenizer vocab size: 57445

Pretrained translation test:
 English: Calculate the voltage across the resistor.
 Afrikaans: Bereken die spanning oor die teenstander.


## Tokenize Datasets


In [23]:
# Tokenize training and validation datasets
MAX_LENGTH = 128

# Uses the MarianMT SentencePiece tokenizer to encode both training and validation data. Max sequence length 128 tokens
def tokenize_pairs(pairs, tok, max_length=MAX_LENGTH):
    # Tokenize a list of {en, af} dicts for seq2seq training.
    en_texts = [p["en"] for p in pairs]
    af_texts = [p["af"] for p in pairs]
    model_inputs = tok(
        en_texts,
        text_target=af_texts,
        max_length=max_length,
        truncation=True,
        padding=False,
    )
    return model_inputs


print("Tokenizing training data")
train_tokenized = tokenize_pairs(train_all, tokenizer)
train_dataset = Dataset.from_dict(train_tokenized)

print("Tokenizing validation data")
val_tokenized = tokenize_pairs(val_provided, tokenizer)
val_dataset = Dataset.from_dict(val_tokenized)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset:   {len(val_dataset)} examples")
print("Sample tokenized input_ids length:", len(train_dataset[0]["input_ids"]))

Tokenizing training data
Tokenizing validation data
Train dataset: 51362 examples
Val dataset:   182 examples
Sample tokenized input_ids length: 3


## Baseline BLEU (Before Fine-tuning)


In [24]:
# Compute baseline BLEU score (pretrained model before fine-tuning)
def translate_batch(en_sentences, mdl, tok, batch_size=16):
    # Translate a list of English sentences to Afrikaans
    translations = []
    mdl.eval()
    with torch.no_grad():
        for i in range(0, len(en_sentences), batch_size):
            batch = en_sentences[i:i+batch_size]
            inputs = tok(batch, return_tensors="pt", padding=True,
                         truncation=True, max_length=MAX_LENGTH)
            outputs = mdl.generate(**inputs, num_beams=4, max_length=MAX_LENGTH)
            decoded = tok.batch_decode(outputs, skip_special_tokens=True)
            translations.extend(decoded)
    return translations


def compute_bleu(hypotheses, references):
    # Compute dataset BLEU using sacrebleu
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score


val_en = [p["en"] for p in val_provided]
val_af = [p["af"] for p in val_provided]

print("Generating baseline translations")
baseline_translations = translate_batch(val_en, model, tokenizer)
baseline_bleu = compute_bleu(baseline_translations, val_af)

print(f"\nBaseline BLEU (pretrained no fine-tuning): {baseline_bleu:.2f}")


Generating baseline translations

Baseline BLEU (pretrained no fine-tuning): 32.63


## Phase 1 Fine-tuning: OPUS + Provided Data

 **This takes 45–90 minutes on CPU.**

In [26]:
# Phase 1 fine-tuning OPUS + provided data combined
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

phase1_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR / "phase1"),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir=str(MODEL_DIR / "logs"),
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=False,
    use_cpu=True,
    seed=SEED,
    report_to="none",
)

phase1_trainer = Seq2SeqTrainer(
    model=model,
    args=phase1_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting Phase 1 training")
phase1_trainer.train()
print("Phase 1 complete")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting Phase 1 training


Epoch,Training Loss,Validation Loss
1,1.063052,1.982368
2,0.773592,2.064754
3,0.578751,2.117750


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.42it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Phase 1 complete


## Phase 2 Fine-tuning: Provided Engineering Data Only
**This takes ~10–30 minutes on CPU.**

In [27]:
# Second training phase on the provided engineering assessment data only
# Lower learning rate for careful domain understanding 

print("Tokenizing provided: only training data for phase 2")
train_provided_tokenized = tokenize_pairs(train_provided * UPSAMPLE_FACTOR, tokenizer)
train_provided_dataset = Dataset.from_dict(train_provided_tokenized)

phase2_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR / "phase2"),
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=5e-5,
    logging_dir=str(MODEL_DIR / "logs"),
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=False,
    use_cpu=True,
    seed=SEED,
    report_to="none",
)

phase2_trainer = Seq2SeqTrainer(
    model=model,
    args=phase2_args,
    train_dataset=train_provided_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting Phase 2 training")
phase2_trainer.train()

model.save_pretrained(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print("Phase 2 complete. Final model saved to model/final/")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Tokenizing provided: only training data for phase 2
Starting Phase 2 training


Epoch,Training Loss,Validation Loss
1,0.120965,2.092792
2,0.045415,2.181583
3,0.028073,2.225708
4,0.015772,2.251246


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.82it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.22it/s]


Phase 2 complete. Final model saved to model/final/


## Post-Fine-tuning BLEU



In [28]:
# Compute BLEU after fine-tuning and save all translations
# Evaluates the fine-tuned model on the val set and compares against the baseline.
print("Generating fine-tuned translations on validation set")
finetuned_translations = translate_batch(val_en, model, tokenizer)
finetuned_bleu = compute_bleu(finetuned_translations, val_af)

print(f"\nBaseline BLEU (pretrained):    {baseline_bleu:.2f}")
print(f"Fine-tuned BLEU:               {finetuned_bleu:.2f}")
print(f"BLEU improvement:              +{finetuned_bleu - baseline_bleu:.2f}")


Generating fine-tuned translations on validation set

Baseline BLEU (pretrained):    32.63
Fine-tuned BLEU:               36.03
BLEU improvement:              +3.40


## Qualitative Analysis


In [29]:
# Cell 11: Qualitative analysis — good and bad translation examples
# Ranks translations by sentence-level BLEU to identify the best and worst examples. Filters out very short sentences (< 5 words) 
def sentence_bleu_score(hypothesis, reference):
    # Compute sentence-level BLEU for ranking
    bleu = sacrebleu.sentence_bleu(hypothesis, [reference])
    return bleu.score

scored = []
for en, af_ref, af_hyp in zip(val_en, val_af, finetuned_translations):
    score = sentence_bleu_score(af_hyp, af_ref)
    scored.append({"en": en, "ref": af_ref, "hyp": af_hyp, "bleu": score})

scored_sorted = sorted(scored, key=lambda x: x["bleu"], reverse=True)
meaningful = [s for s in scored_sorted if len(s["en"].split()) >= 5]
best  = meaningful[:3]
worst = meaningful[-3:]

print("=" * 70)
print("GOOD TRANSLATIONS (high sentence BLEU)")
print("=" * 70)
for i, ex in enumerate(best, 1):
    print(f"\nExample {i} (BLEU: {ex['bleu']:.1f})")
    print(f"  EN:  {ex['en']}")
    print(f"  REF: {ex['ref']}")
    print(f"  HYP: {ex['hyp']}")

print("\n" + "=" * 70)
print("BAD TRANSLATIONS (low sentence BLEU)")
print("=" * 70)
for i, ex in enumerate(worst, 1):
    print(f"\nExample {i} (BLEU: {ex['bleu']:.1f})")
    print(f"  EN:  {ex['en']}")
    print(f"  REF: {ex['ref']}")
    print(f"  HYP: {ex['hyp']}")


GOOD TRANSLATIONS (high sentence BLEU)

Example 1 (BLEU: 100.0)
  EN:  Give a reason for your answer.
  REF: Gee 'n rede vir jou antwoord.
  HYP: Gee 'n rede vir jou antwoord.

Example 2 (BLEU: 100.0)
  EN:  What is the function of JTAG?
  REF: Wat is die funksie van JTAG?
  HYP: Wat is die funksie van JTAG?

Example 3 (BLEU: 100.0)
  EN:  Give a reason for your answer.
  REF: Gee 'n rede vir jou antwoord.
  HYP: Gee 'n rede vir jou antwoord.

BAD TRANSLATIONS (low sentence BLEU)

Example 1 (BLEU: 4.6)
  EN:  The output given below is seen from the BB.
  REF: Die onderstaande uitree word gesien met die gebruik van 'n BB.
  HYP: Die uittree wat hier onder gegee word, word van die BB gesien.

Example 2 (BLEU: 4.5)
  EN:  Below is a section of assembler code for the ARM processor.
  REF: 'n Instruksielys van 'n ARM-verwerker word gelys.
  HYP: Onderste is 'n deel van drom kode vir die ARM verwerker.

Example 3 (BLEU: 3.7)
  EN:  The CanSat is launched from a rocket and its mission time mu